# Proyecto final · 04 · Gobernanza

**Componente 3 · Sección 7 del examen.**

Lo que se evalúa:
- Privilegios con mínimo privilegio para un perfil de lectura (`account users` o tu segunda identidad), verificados con `SHOW GRANTS`.
- Un row filter o un column mask controlado por una tabla de mapeo, como en el Laboratorio 4, con la evidencia de lo que ve tu usuario antes y después de cambiar el mapeo.
- Tags en las columnas sensibles.

In [0]:
-- Seleccionar el catálogo del proyecto para configurar su gobernanza.
USE CATALOG pf_fallas;

In [0]:
-- Permitir al perfil de lectura navegar hasta Gold y consultar su resumen.
-- No otorgar escritura ni acceso a Bronze, Silver o la tabla de permisos.
GRANT USE CATALOG ON CATALOG pf_fallas TO `account users`;

GRANT USE SCHEMA ON SCHEMA pf_fallas.gold TO `account users`;

GRANT SELECT ON TABLE pf_fallas.gold.vehiculos_por_fabricante
TO `account users`;

In [0]:
-- Revisar los permisos del catálogo, del esquema Gold y de su tabla.
SHOW GRANTS ON CATALOG pf_fallas;

SHOW GRANTS ON SCHEMA pf_fallas.gold;

SHOW GRANTS ON TABLE pf_fallas.gold.vehiculos_por_fabricante;

Principal,ActionType,ObjectType,ObjectKey
account users,SELECT,TABLE,pf_fallas.gold.vehiculos_por_fabricante


In [0]:
-- Crear un esquema de seguridad y una tabla que determine
-- qué usuarios pueden consultar el VIN parcial sin enmascarar.
CREATE SCHEMA IF NOT EXISTS pf_fallas.seguridad;

CREATE TABLE IF NOT EXISTS pf_fallas.seguridad.acceso_vin (
    usuario STRING,
    puede_ver_vin BOOLEAN
)
USING DELTA;

In [0]:
-- Registrar al usuario actual con permiso para ver el VIN parcial.
-- MERGE evita duplicar su registro si esta celda vuelve a ejecutarse.
MERGE INTO pf_fallas.seguridad.acceso_vin AS destino
USING (
    SELECT current_user() AS usuario, true AS puede_ver_vin
) AS origen
ON destino.usuario = origen.usuario
WHEN MATCHED THEN
    UPDATE SET destino.puede_ver_vin = origen.puede_ver_vin
WHEN NOT MATCHED THEN
    INSERT (usuario, puede_ver_vin)
    VALUES (origen.usuario, origen.puede_ver_vin);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
-- Mostrar el VIN parcial solamente a usuarios autorizados
-- en la tabla de mapeo; ocultarlo para todos los demás.
CREATE OR REPLACE FUNCTION pf_fallas.seguridad.enmascarar_vin(
    valor STRING
)
RETURNS STRING
RETURN
    CASE
        WHEN EXISTS (
            SELECT 1
            FROM pf_fallas.seguridad.acceso_vin
            WHERE usuario = current_user()
              AND puede_ver_vin = true
        )
        THEN valor
        ELSE 'RESTRINGIDO'
    END;

ALTER TABLE pf_fallas.silver.vehiculos
ALTER COLUMN vin_prefix
SET MASK pf_fallas.seguridad.enmascarar_vin;

In [0]:
-- Verificar que el usuario autorizado puede ver el VIN parcial.
SELECT
    current_user() AS usuario,
    vehicle_id,
    vin_prefix,
    make
FROM pf_fallas.silver.vehiculos
ORDER BY vehicle_id
LIMIT 5;

usuario,vehicle_id,vin_prefix,make
josuefallas@gmail.com,4469,1G1RB6S51J,CHEVROLET
josuefallas@gmail.com,4777,5YJXCBE49G,TESLA
josuefallas@gmail.com,10734,1N4AZ0CP1D,NISSAN
josuefallas@gmail.com,24629,5YJ3E1EA2J,TESLA
josuefallas@gmail.com,90772,JN1AZ0CP6C,NISSAN


In [0]:
-- Quitar al usuario actual la autorización para ver el VIN parcial.
-- La máscara seguirá aplicada y consultará este nuevo permiso.
UPDATE pf_fallas.seguridad.acceso_vin
SET puede_ver_vin = false
WHERE usuario = current_user();

num_affected_rows
1


In [0]:
-- Comprobar que el VIN parcial aparece como RESTRINGIDO
-- cuando el usuario no tiene autorización en la tabla de mapeo.
SELECT
    current_user() AS usuario,
    vehicle_id,
    vin_prefix,
    make
FROM pf_fallas.silver.vehiculos
ORDER BY vehicle_id
LIMIT 5;

usuario,vehicle_id,vin_prefix,make
josuefallas@gmail.com,4469,RESTRINGIDO,CHEVROLET
josuefallas@gmail.com,4777,RESTRINGIDO,TESLA
josuefallas@gmail.com,10734,RESTRINGIDO,NISSAN
josuefallas@gmail.com,24629,RESTRINGIDO,TESLA
josuefallas@gmail.com,90772,RESTRINGIDO,NISSAN


In [0]:
-- Restaurar el permiso del usuario actual para consultar el VIN parcial.
UPDATE pf_fallas.seguridad.acceso_vin
SET puede_ver_vin = true
WHERE usuario = current_user();

num_affected_rows
1


In [0]:
-- Comprobar que los valores originales vuelven a mostrarse
-- después de restaurar la autorización en la tabla de mapeo.
SELECT
    current_user() AS usuario,
    vehicle_id,
    vin_prefix,
    make
FROM pf_fallas.silver.vehiculos
ORDER BY vehicle_id
LIMIT 5;

usuario,vehicle_id,vin_prefix,make
josuefallas@gmail.com,4469,1G1RB6S51J,CHEVROLET
josuefallas@gmail.com,4777,5YJXCBE49G,TESLA
josuefallas@gmail.com,10734,1N4AZ0CP1D,NISSAN
josuefallas@gmail.com,24629,5YJ3E1EA2J,TESLA
josuefallas@gmail.com,90772,JN1AZ0CP6C,NISSAN


In [0]:
-- Documentar la clasificación de los identificadores del proyecto.
-- Estos tags son descriptivos y no aplican permisos por sí mismos.
-- Usaremos un tag propio, clasificacion_proyecto, para documentar el tratamiento de los identificadores
ALTER TABLE pf_fallas.silver.vehiculos
ALTER COLUMN vin_prefix
SET TAGS ('clasificacion_proyecto' = 'identificador_restringido');

ALTER TABLE pf_fallas.silver.vehiculos
ALTER COLUMN vehicle_id
SET TAGS ('clasificacion_proyecto' = 'identificador_vehiculo');

In [0]:
-- Consultar las etiquetas de las columnas para conservar
-- evidencia de la clasificación aplicada en Silver.
-- Verificar los tags
SELECT
    table_name,
    column_name,
    tag_name,
    tag_value
FROM pf_fallas.information_schema.column_tags
WHERE schema_name = 'silver'
  AND table_name = 'vehiculos'
ORDER BY column_name, tag_name;

table_name,column_name,tag_name,tag_value
vehiculos,vehicle_id,clasificacion_proyecto,identificador_vehiculo
vehiculos,vin_prefix,clasificacion_proyecto,identificador_restringido


In [0]:
-- Comprobar que la máscara del VIN parcial sigue aplicada
-- después de volver a ejecutar el pipeline.
SELECT *
FROM pf_fallas.information_schema.column_masks
WHERE table_schema = 'silver'
  AND table_name = 'vehiculos';

table_catalog,table_schema,table_name,column_name,mask_name,using_columns
pf_fallas,silver,vehiculos,vin_prefix,pf_fallas.seguridad.enmascarar_vin,


In [0]:
-- Comprobar que las etiquetas de clasificación permanecen
-- después de actualizar la tabla Silver.
SELECT
    column_name,
    tag_name,
    tag_value
FROM pf_fallas.information_schema.column_tags
WHERE schema_name = 'silver'
  AND table_name = 'vehiculos'
ORDER BY column_name;

column_name,tag_name,tag_value
vehicle_id,clasificacion_proyecto,identificador_vehiculo
vin_prefix,clasificacion_proyecto,identificador_restringido


### TU RESPUESTA
¿Quién ve qué en tu proyecto? Explícalo en tres líneas, como se lo explicarías a un auditor.

El perfil `account users` recibió permisos para consultar el resumen Gold,
sin otorgarle escritura ni acceso a Bronze, Silver o la tabla de mapeo.

En Silver, `vin_prefix` se muestra únicamente cuando el usuario tiene
`puede_ver_vin = true` en la tabla `seguridad.acceso_vin`; de lo contrario,
aparece como `RESTRINGIDO`.

Los identificadores tienen tags descriptivos. Se verificó que tanto los tags
como la máscara permanecen después de ejecutar nuevamente el Job.